# pips

In [2]:
# !pip install faiss-cpu

# !pip install openai

# load data

In [3]:



import os
import pickle
import io
import torch

# Custom unpickler that forces CUDA tensors to load on CPU
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if module == 'torch.storage' and name == '_load_from_bytes':
            # Force loading all storages to CPU
            return lambda b: torch.load(io.BytesIO(b), map_location=torch.device('cpu'))
        return super().find_class(module, name)

class DataStore:
    # Constant keys for variable names
    KEY_TEST_PROMPTS       = "test_prompts"
    KEY_MODEL_FEATURES     = "model_features_matrix"
    KEY_MODEL_ACTIVATIONS  = "model_activations_matrix"
    KEY_NTK                = "ntk_matrix"
    KEY_SAE_AGG            = "sae_aggregation_matrix"
    KEY_ACTIVATION_AGG     = "activation_aggregation_matrix"
    KEY_TRUE_LABELS        = "true_labels"

    def __init__(self, test_prompts, model_features_matrix, model_activations_matrix,
                 ntk_matrix, sae_aggregation_matrix, activation_aggregation_matrix, true_labels):
        self.test_prompts = test_prompts
        self.model_features_matrix = model_features_matrix
        self.model_activations_matrix = model_activations_matrix
        self.ntk_matrix = ntk_matrix
        self.sae_aggregation_matrix = sae_aggregation_matrix
        self.activation_aggregation_matrix = activation_aggregation_matrix
        self.true_labels = true_labels

    def save(self, filename="data/input_variables.pkl"):
        # Ensure the directory exists
        os.makedirs(os.path.dirname(filename), exist_ok=True)
        
        # Prepare data for saving; convert any tensors to CPU to avoid CUDA issues.
        data = {
            DataStore.KEY_TEST_PROMPTS: self.test_prompts,
            DataStore.KEY_MODEL_FEATURES: self.model_features_matrix.cpu() if hasattr(self.model_features_matrix, 'cpu') else self.model_features_matrix,
            DataStore.KEY_MODEL_ACTIVATIONS: self.model_activations_matrix.cpu() if hasattr(self.model_activations_matrix, 'cpu') else self.model_activations_matrix,
            DataStore.KEY_NTK: self.ntk_matrix.cpu() if hasattr(self.ntk_matrix, 'cpu') else self.ntk_matrix,
            DataStore.KEY_SAE_AGG: self.sae_aggregation_matrix.cpu() if hasattr(self.sae_aggregation_matrix, 'cpu') else self.sae_aggregation_matrix,
            DataStore.KEY_ACTIVATION_AGG: self.activation_aggregation_matrix.cpu() if hasattr(self.activation_aggregation_matrix, 'cpu') else self.activation_aggregation_matrix,
            DataStore.KEY_TRUE_LABELS: self.true_labels,
        }
        
        # Save using pickle with the highest protocol.
        with open(filename, "wb") as f:
            pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Variables saved to {filename}")

    @classmethod
    def load(cls, filename="data/input_variables.pkl"):
        # Use CPU_Unpickler to load data and force CUDA storages to CPU
        with open(filename, "rb") as f:
            data = CPU_Unpickler(f).load()
        
        # Helper to ensure tensors are on CPU
        def maybe_to_device(t):
            return t.to(torch.device("cpu")) if hasattr(t, "to") else t
        
        print(f"Variables loaded from {filename} using CPU_Unpickler.")
        return cls(
            test_prompts=data[cls.KEY_TEST_PROMPTS],
            model_features_matrix=maybe_to_device(data[cls.KEY_MODEL_FEATURES]),
            model_activations_matrix=maybe_to_device(data[cls.KEY_MODEL_ACTIVATIONS]),
            ntk_matrix=maybe_to_device(data[cls.KEY_NTK]),
            sae_aggregation_matrix=maybe_to_device(data[cls.KEY_SAE_AGG]),
            activation_aggregation_matrix=maybe_to_device(data[cls.KEY_ACTIVATION_AGG]),
            true_labels=data[cls.KEY_TRUE_LABELS],
        )

    def get_keys(self):
        """
        Returns a list of all constant keys used for storing variables.
        """
        return [
            self.KEY_TEST_PROMPTS,
            self.KEY_MODEL_FEATURES,
            self.KEY_MODEL_ACTIVATIONS,
            self.KEY_NTK,
            self.KEY_SAE_AGG,
            self.KEY_ACTIVATION_AGG,
            self.KEY_TRUE_LABELS
        ]

    def expand_to_namespace(self, namespace):
        """
        Injects the stored variables into the provided namespace dictionary.
        """
        namespace[self.KEY_TEST_PROMPTS] = self.test_prompts
        namespace[self.KEY_MODEL_FEATURES] = self.model_features_matrix
        namespace[self.KEY_MODEL_ACTIVATIONS] = self.model_activations_matrix
        namespace[self.KEY_NTK] = self.ntk_matrix
        namespace[self.KEY_SAE_AGG] = self.sae_aggregation_matrix
        namespace[self.KEY_ACTIVATION_AGG] = self.activation_aggregation_matrix
        namespace[self.KEY_TRUE_LABELS] = self.true_labels
        for k in self.get_keys():
            print(k)
        print("Variables expanded to the provided namespace.")




# # Assuming your variables already exist:
# datastore = DataStore(
#     test_prompts, model_features_matrix, model_activations_matrix,
#     ntk_matrix, sae_aggregation_matrix, activation_aggregation_matrix, true_labels
# )
# datastore.save()  # Saves to data/input_variables.pkl






datastore = DataStore.load()  # Loads from data/input_variables.pkl
datastore.expand_to_namespace(globals())
# Now you can directly access:
# test_prompts, model_features_matrix, model_activations_matrix, ..., true_labels



Variables loaded from data/input_variables.pkl using CPU_Unpickler.
test_prompts
model_features_matrix
model_activations_matrix
ntk_matrix
sae_aggregation_matrix
activation_aggregation_matrix
true_labels
Variables expanded to the provided namespace.


# utils

In [35]:
import numpy as np
import faiss

import platform
def get_device(skip_apple_silicon=False):
    if torch.cuda.is_available():
        device = "cuda"
        device_name = f"CUDA GPU ({torch.cuda.get_device_name(0)})"
    elif platform.processor() == 'arm' and platform.system() == 'Darwin' and not skip_apple_silicon:
        device = "mps"
        device_name = "Apple Silicon (MPS)"
    else:
        device = "cpu"
        device_name = "CPU"
    return device, device_name


# --- Utility: Normalize a Matrix ---
def normalize_matrix(M: np.ndarray):
    M_tensor = torch.tensor(M, device=device)
    diag = torch.sqrt(torch.diag(M_tensor))
    norm_matrix = M_tensor / (diag.unsqueeze(1) * diag.unsqueeze(0) + 1e-8)
    return norm_matrix.detach().cpu().float().numpy()
    

def get_similarity_matrices(sae_agg, act_agg, ntk):
    """
    Compute and normalize similarity matrices from aggregation matrices.
    Assumes inputs are torch tensors.
    """
    # Compute dependency graphs from aggregation matrices
    sae_dep = (sae_agg @ sae_agg.T).cpu().float().numpy()
    act_dep = (act_agg @ act_agg.T).cpu().float().numpy()
    
    # Normalize the matrices (assumes normalize_matrix is defined)
    norm_sae = normalize_matrix(sae_dep)
    norm_act = normalize_matrix(act_dep)
    norm_ntk = normalize_matrix(ntk)
    
    return {
        "SAE Similarity": norm_sae,
        "Activation Similarity": norm_act,
        "NTK Similarity": norm_ntk
    }


    
# Import the model
device, device_name = get_device(skip_apple_silicon=False)



# generate matrices

In [35]:
for i, prompt in enumerate(test_prompts):
    sae_aggregation_matrix[i] = torch.nn.functional.normalize(model_features_matrix[i], p=2, dim=0).mean(dim=0)
    activation_aggregation_matrix[i] = torch.nn.functional.normalize(model_activations_matrix[i], p=2, dim=0).mean(dim=0)

# Get normalized similarity matrices.
matrices = get_similarity_matrices(sae_aggregation_matrix,
                                   activation_aggregation_matrix,
                                   ntk_matrix)

# precompute_similarity_ordering

In [35]:
import numpy as np

def precompute_similarity_orderings(matrices):
    """
    Given a dictionary of similarity matrices (keys like "SAE Similarity", etc.),
    compute and return a new dictionary with the same keys where each value is a dict 
    containing 'sorted_indices' and 'sorted_scores'. Each row in the matrices is 
    sorted in descending order (i.e., the instance itself will be first).
    
    Parameters:
      matrices (dict): Dictionary where keys are names of similarity matrices and
                       values are numpy arrays of shape (n_instances, n_instances)
    
    Returns:
      dict: Dictionary with the same keys. For each key, a dictionary with:
            - 'sorted_indices': indices sorted by descending similarity per row.
            - 'sorted_scores': corresponding sorted similarity scores.
    """
    ordered_results = {}
    for key, matrix in matrices.items():
        # Ensure the matrix is a NumPy array.
        mat = np.array(matrix)
        
        # Sort indices in descending order (largest similarity first)
        sorted_indices = np.argsort(-mat, axis=1)
        # Sort the scores accordingly. Use advanced indexing.
        sorted_scores = np.take_along_axis(mat, sorted_indices, axis=1)
        
        ordered_results[key] = {
            'sorted_indices': sorted_indices,
            'sorted_scores': sorted_scores
        }
    return ordered_results

# Example usage:
# Assume 'matrices' is a dictionary produced by get_similarity_matrices(...)
# matrices = get_similarity_matrices(sae_aggregation_matrix, activation_aggregation_matrix, ntk_matrix)
# Precompute the orderings for coarse-graining.
similarity_orderings = precompute_similarity_orderings(matrices)

# Now similarity_orderings has the same keys as matrices.
# For example, to get the sorted indices and scores for the NTK Similarity:
ntk_order = similarity_orderings["NTK Similarity"]
print("First 5 indices (for query 0):", ntk_order['sorted_indices'][0][:5])
print("Corresponding scores:", ntk_order['sorted_scores'][0][:5])


First 5 indices (for query 0): [  0 140 185 158 106]
Corresponding scores: [1.         0.14529322 0.1430642  0.13210574 0.12452891]


# top_N_extracter

In [35]:
# def get_top_neighbors(ordering, query_idx, num_neighbors, prompts, labels, print_output=False):
#     """
#     For a given similarity ordering (with 'sorted_indices' and 'sorted_scores'),
#     return the top neighbors as three lists:
#       - neighbor texts
#       - neighbor labels
#       - neighbor scores
      
#     Parameters:
#       ordering (dict): Dictionary containing 'sorted_indices' and 'sorted_scores' as NumPy arrays.
#       query_idx (int): Index of the query instance.
#       num_neighbors (int): Number of top neighbors to return.
#       prompts (list): List of strings (e.g., prompt texts) for each instance.
#       labels (list): List of labels corresponding to each instance.
#       print_output (bool, optional): If True, prints the results. Defaults to False.
      
#     Returns:
#       tuple: (neighbor_texts, neighbor_labels, neighbor_scores)
#     """
#     sorted_indices = ordering['sorted_indices']
#     sorted_scores = ordering['sorted_scores']

#     # Retrieve the top neighbor indices and scores for the query instance.
#     top_indices = sorted_indices[query_idx, :num_neighbors]
#     top_scores = sorted_scores[query_idx, :num_neighbors]

#     # Build the corresponding lists for texts and labels.
#     neighbor_texts = [prompts[i] for i in top_indices]
#     neighbor_labels = [labels[i] if labels is not None else "N/A" for i in top_indices]
#     neighbor_scores = top_scores.tolist()

#     if print_output:
#         print(f"--- Top Neighbors for Query Index {query_idx} ---")
#         for rank, (idx, score, text, lab) in enumerate(zip(top_indices, neighbor_scores, neighbor_texts, neighbor_labels), 1):
#             print(f"{rank}. Index: {idx}, Score: {score:.4f}, Label: {lab}")
#             print(f"   Text: {text}\n")

#     return neighbor_texts, neighbor_labels, neighbor_scores

# # # Example usage:
# neighbor_texts, neighbor_labels, neighbor_scores = get_top_neighbors(similarity_orderings["SAE Similarity"], 
#                                                                      query_idx=0, 
#                                                                      num_neighbors=5, 
#                                                                      prompts=test_prompts, 
#                                                                      labels=true_labels,
#                                                                      print_output=False
#                                                                     )


# get_semantic V4 with parallel

In [35]:
import random
import os
import pickle
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Create data folder if it doesn't exist.
DATA_DIR = "data"
if not os.path.exists(DATA_DIR):
    os.makedirs(DATA_DIR)

# Cache file name.
CACHE_FILE = os.path.join(DATA_DIR, "course_graining_results.pkl")

# Load environment variables from .env file (if available)
load_dotenv()

# Option 1: Use environment variable (recommended)
api_key = os.getenv("OPENAI_API_KEY")
# Option 2: Direct assignment (uncomment if not using env vars)
# api_key = 'your-api-key-here'

if api_key is None:
    raise ValueError("Please set your OPENAI_API_KEY environment variable or assign it directly in the code.")

# Instantiate the OpenAI client with the new API interface
client = OpenAI(api_key=api_key)

# Default API configuration
API_CONFIG = {
    "model": "gpt-4o-mini",  # adjust to your desired model
    "temperature": 0.02,
    "max_tokens": 10000,
}

def select_indices_by_label(labels, count, random_seed=None):
    """
    Given a list of integer labels, randomly select `count` indices for each unique label.
    The selected indices for each label are sorted in ascending order, and the final list
    is ordered such that all indices from the lowest label appear first, then the next label, etc.
    
    Parameters:
        labels (list of int): The list of labels.
        count (int): Number of indices to select per unique label.
        random_seed (int, optional): Seed for reproducibility.
        
    Returns:
        list: A list of selected indices.
    """
    if random_seed is not None:
        random.seed(random_seed)
    
    label_to_indices = {}
    for idx, lab in enumerate(labels):
        label_to_indices.setdefault(lab, []).append(idx)
    
    selected_indices = []
    for lab in sorted(label_to_indices.keys()):
        indices = label_to_indices[lab]
        if len(indices) > count:
            selected = random.sample(indices, count)
        else:
            selected = indices
        selected_indices.extend(sorted(selected))
    
    return selected_indices

def get_semantic_analysis(strings, n_tags=3, api_config=API_CONFIG, progress_bar=None):
    """
    Given a list of semantically grouped strings, generate:
      1. A detailed semantic description.
      2. A short summary wrapped in 10 hash symbols, which is then parsed into a list.
    
    Returns:
      tuple: (detailed_description, extracted_tags_list)
    """
    conversation_history = [
        {"role": "system", "content": "You are an AI assistant skilled in semantic analysis."}
    ]
    user_input_description = (
        "Here are some strings grouped based on semantic similarity:\n\n" +
        "\n".join(strings) +
        "\n\nPlease provide a detailed description of the common semantic aspects among these strings."
    )
    conversation_history.append({"role": "user", "content": user_input_description})
    
    response_description = client.chat.completions.create(
        model=api_config["model"],
        messages=conversation_history,
        temperature=api_config["temperature"],
        max_tokens=api_config["max_tokens"]
    )
    detailed_description = response_description.choices[0].message.content.strip()
    if progress_bar is not None:
        progress_bar.update(1)
    
    conversation_history.append({"role": "assistant", "content": detailed_description})
    user_input_tags = (
        f"Based on the above description and provided strings, if you had to choose {n_tags} words or short phrases "
        "to encapsulate the common themes, what would they be? "
        "Please format your answer exactly as follows:\n"
        "##########word1, word2, word3##########"
    )
    conversation_history.append({"role": "user", "content": user_input_tags})
    
    response_tags = client.chat.completions.create(
        model=api_config["model"],
        messages=conversation_history,
        temperature=api_config["temperature"],
        max_tokens=api_config["max_tokens"]
    )
    formatted_tags = response_tags.choices[0].message.content.strip()
    if progress_bar is not None:
        progress_bar.update(1)
    
    if formatted_tags.startswith("##########") and formatted_tags.endswith("##########"):
        tag_str = formatted_tags[len("##########"):-len("##########")]
        extracted_tags = [tag.strip() for tag in tag_str.split(",")]
    else:
        extracted_tags = [formatted_tags]
    
    return detailed_description, extracted_tags

def get_top_neighbors(ordering, query_idx, num_neighbors, prompts, labels, print_output=False):
    """
    For a given ordering (with 'sorted_indices' and 'sorted_scores'), return the top neighbors.
    
    Returns:
      tuple: (neighbor_texts, neighbor_labels, neighbor_scores, neighbor_indices)
    """
    sorted_indices = ordering['sorted_indices']
    sorted_scores = ordering['sorted_scores']
    top_indices = sorted_indices[query_idx, :num_neighbors]
    top_scores = sorted_scores[query_idx, :num_neighbors]
    neighbor_texts = [prompts[i] for i in top_indices]
    neighbor_labels = [labels[i] if labels is not None else "N/A" for i in top_indices]
    neighbor_scores = top_scores.tolist()
    neighbor_indices = top_indices.tolist()  # include indices for downstream processing
    
    if print_output:
        print(f"--- Top Neighbors for Query Index {query_idx} ---")
        for rank, (idx, score, text, lab) in enumerate(zip(top_indices, neighbor_scores, neighbor_texts, neighbor_labels), 1):
            print(f"{rank}. Index: {idx}, Score: {score:.4f}, Label: {lab}")
            print(f"   Text: {text}\n")
    
    return neighbor_texts, neighbor_labels, neighbor_scores, neighbor_indices




from datetime import datetime
from concurrent.futures import ThreadPoolExecutor
import os
import pickle

def get_cache_file_path(custom_label):
    """
    Returns a cache file path with the current date prepended.
    Format: YYYY-MM-DD_{custom_label}.pkl
    """
    date_str = datetime.now().strftime("%Y-%m-%d")
    file_name = f"{date_str}_{custom_label}.pkl"
    return os.path.join(DATA_DIR, file_name)

def process_all_similarity_orderings_with_cache(similarity_orderings, query_indices, neighbor_counts,
                                                test_prompts, true_labels, n_tags=3, print_output=False,
                                                api_config=API_CONFIG, max_simultaneous_chats=10,
                                                experiment_save_name=None, load_experiment=None):
    """
    Processes each similarity ordering for each query index and neighbor count, using a cache file.
    
    You must specify exactly one of:
      - experiment_save_name: a custom string to start a new experiment run.
      - load_experiment: the exact filename (including .pkl) of an existing cache to load.
    
    If experiment_save_name is provided, a new cache is created using a file name that
    prepends today's date (e.g., "2025-03-27_custom_label.pkl"). Any existing file with that name is ignored.
    
    If load_experiment is provided, that exact file is loaded and updated as needed.
    
    Raises:
      ValueError: if both or neither of experiment_save_name and load_experiment are provided.
    """
    # Validate input parameters.
    if (experiment_save_name is None and load_experiment is None) or (experiment_save_name is not None and load_experiment is not None):
        raise ValueError("Please specify exactly one: either experiment_save_name (for a new run) or load_experiment (to load an existing file).")
    
    # Determine cache file path.
    if experiment_save_name is not None:
        cache_file = get_cache_file_path(experiment_save_name)
        cache = {}  # Starting new; ignore any existing file.
    else:
        # load_experiment is not None.
        cache_file = load_experiment  # Use the exact filename provided.
        if not os.path.exists(cache_file):
            raise FileNotFoundError(f"Specified load_experiment file '{cache_file}' does not exist.")
        with open(cache_file, "rb") as f:
            cache = pickle.load(f)

    results = {}
    total_orderings = len(similarity_orderings)
    total_queries = len(query_indices)
    total_counts = len(neighbor_counts)
    total_api_calls = total_orderings * total_queries * total_counts * 2
    print(f"Total API calls (if no caching): {total_api_calls}")

    futures = {}
    executor = ThreadPoolExecutor(max_workers=max_simultaneous_chats)

    with tqdm(total=total_api_calls, desc="Overall API Progress") as pbar:
        for ordering_key, ordering in similarity_orderings.items():
            if ordering_key not in cache:
                cache[ordering_key] = {}
            ordering_result = {}
            for query_idx in query_indices:
                if query_idx not in cache[ordering_key]:
                    cache[ordering_key][query_idx] = {}
                query_result = {}
                # Save self info (wrap text and label in a list)
                self_info = {
                    "neighbor_texts": [test_prompts[query_idx]],
                    "neighbor_labels": [true_labels[query_idx]],
                    "neighbor_scores": 1.0,
                    "neighbor_indices": query_idx
                }
                query_result["self"] = self_info
                for count in neighbor_counts:
                    key = f"N_{count}"
                    if key in cache[ordering_key][query_idx]:
                        query_result[key] = cache[ordering_key][query_idx][key]
                    else:
                        # Get top neighbor info including indices.
                        neighbor_texts, neighbor_labels, neighbor_scores, neighbor_indices = get_top_neighbors(
                            ordering, query_idx, count, test_prompts, true_labels, print_output=False
                        )
                        # Schedule the API call concurrently.
                        future = executor.submit(get_semantic_analysis, neighbor_texts, n_tags, api_config, pbar)
                        futures[(ordering_key, query_idx, key)] = (future, neighbor_texts, neighbor_labels, neighbor_scores, neighbor_indices)
                ordering_result[query_idx] = query_result
            results[ordering_key] = ordering_result

        # Wait for all API calls to complete.
        for (ordering_key, query_idx, key), (future, neighbor_texts, neighbor_labels, neighbor_scores, neighbor_indices) in futures.items():
            detailed_description, extracted_tags = future.result()
            result_entry = {
                "neighbor_texts": neighbor_texts,
                "neighbor_labels": neighbor_labels,
                "neighbor_scores": neighbor_scores,
                "neighbor_indices": neighbor_indices,
                # Wrap the detailed description in a list.
                "detailed_description": [detailed_description],
                "extracted_tags": extracted_tags,
            }
            cache[ordering_key][query_idx][key] = result_entry
            results[ordering_key][query_idx][key] = result_entry
    executor.shutdown()

    # Save the updated cache to the determined file.
    with open(cache_file, "wb") as f:
        pickle.dump(cache, f)

    return results




#NOTE SETTING need to set either the experiment name or the laod experimetn file. 
query_indices = select_indices_by_label(true_labels, count=10, random_seed=42)
experiment_save_name = "200_News_titles_V2"
experiment_save_name = None

bd = "/Users/phillipmaire/Documents/GitHub/ds_v3/notebooks/PIBBS-coarse-graining/data/"
load_experiment = bd+"2025-03-31_200_News_titles_V2.pkl"
# load_experiment = None



course_graining_results = process_all_similarity_orderings_with_cache(
    similarity_orderings=similarity_orderings,
    query_indices=query_indices,
    neighbor_counts=[5, 10, 20, 40, 80],
    test_prompts=test_prompts,
    true_labels=true_labels,
    n_tags=3,
    print_output=True,
    api_config=API_CONFIG,
    max_simultaneous_chats=10,
    experiment_save_name=experiment_save_name,
    load_experiment=load_experiment
)




Total API calls (if no caching): 300


Overall API Progress:   0%|          | 0/300 [00:00<?, ?it/s]

# display

In [36]:
from IPython.display import JSON, display


display(JSON(course_graining_results))




<IPython.core.display.JSON object>

# visualize coarse graining

In [37]:

import pandas as pd

def visualize_extracted_tags_stacked_rows(results, target_index, level_keys, num_tags=3):
    """
    For a given target index, creates a DataFrame where each row corresponds to a main key with a tag position appended 
    (e.g. "Dummy Similarity_1", "Dummy Similarity_2", etc.). The columns are the neighbor levels (e.g. "N_5", "N_10", etc.)
    and each cell contains the extracted tag for that main key and level at the given tag position.

    Parameters:
      results (dict): The course graining results dictionary.
      target_index (int): The specific index to inspect.
      level_keys (list of str): The list of neighbor level keys to consider (e.g., ["N_5", "N_10", ...]).
      num_tags (int): The number of expected tags per level.

    Returns:
      pd.DataFrame: A DataFrame with rows named as "<Main Key>_<Tag Position>" and columns for each neighbor level.
    """
    rows = []
    index_labels = []
    
    for main_key, main_data in results.items():
        # For the target index, get results for each level, if available.
        for tag_pos in range(num_tags):
            row = {}
            row_label = f"{main_key}_{tag_pos+1}"
            index_labels.append(row_label)
            for level_key in level_keys:
                # If the data exists for the level, try to extract the tag at the given position.
                if target_index in main_data and level_key in main_data[target_index]:
                    tags = main_data[target_index][level_key].get("extracted_tags", [])
                    row[level_key] = tags[tag_pos] if tag_pos < len(tags) else ""
                else:
                    row[level_key] = ""
            rows.append(row)
    
    df = pd.DataFrame(rows, index=index_labels)
    df.index.name = "Main Key & Tag Position"
    # Order the columns as provided in level_keys.
    df = df[level_keys]
    return df

# ------------------ Run Example ------------------

# Assuming course_graining_results is generated from process_all_similarity_orderings_with_cache,
# and test_prompts is a list of prompt strings.
target_indices = [6, 26, 70, 163, 189]
level_keys = ["N_5", "N_10", "N_20", "N_40", "N_80"]

for target_index in target_indices:
    print("===================================================")
    print(f"Target Index: {target_index}")
    # Print the main string (query prompt) once (truncated to 250 characters).
    main_string = test_prompts[target_index][:250]
    print("Main String (Query Prompt):")
    print(main_string)
    print("---------------------------------------------------")
    
    # Build and display the DataFrame.
    stacked_rows_df = visualize_extracted_tags_stacked_rows(course_graining_results, target_index, level_keys, num_tags=3)
    display(stacked_rows_df)
    print("________________")


Target Index: 6
Main String (Query Prompt):
Donald Trump Fires Up Birther Conspiracy About Marco Rubio
---------------------------------------------------


,N_5,N_10,N_20,N_40,N_80
Main Key & Tag Position,,,,,
SAE Similarity_1,political conflict,political controversy,political controversy,political discourse,political discourse
SAE Similarity_2,media influence,public reaction,public reaction,media reaction,social issues
SAE Similarity_3,accountability,social movements,media engagement,social activism,media influence
Activation Similarity_1,political controversy,political controversy,political controversy,political discourse,political discourse
Activation Similarity_2,media reaction,public reaction,public reaction,social justice,social issues
Activation Similarity_3,Trump dynamics,media scrutiny,policy discussion,media reaction,media reaction
NTK Similarity_1,political discourse,Celebrity Culture,politics,politics,politics
NTK Similarity_2,public figures,Political Discourse,personal narratives,celebrity culture,social issues
NTK Similarity_3,personal choices,Personal Experiences,social issues,social issues,celebrity culture


________________
Target Index: 26
Main String (Query Prompt):
Tyra Banks Graduates From Harvard, Beyoncé's New House Of Dereon Ad And More Style News
---------------------------------------------------


,N_5,N_10,N_20,N_40,N_80
Main Key & Tag Position,,,,,
SAE Similarity_1,celebrity culture,celebrity culture,celebrity culture,celebrity culture,celebrity culture
SAE Similarity_2,social commentary,fashion trends,social advocacy,social issues,political commentary
SAE Similarity_3,entertainment news,political commentary,political commentary,personal relationships,social issues
Activation Similarity_1,celebrity culture,celebrity culture,celebrity culture,celebrity culture,celebrity culture
Activation Similarity_2,personal narratives,social issues,social issues,social issues,social activism
Activation Similarity_3,social issues,entertainment humor,personal narratives,personal narratives,personal narratives
NTK Similarity_1,celebrity fashion,celebrity culture,celebrity culture,celebrity culture,celebrity culture
NTK Similarity_2,style moments,fashion moments,fashion trends,lifestyle trends,social commentary
NTK Similarity_3,cultural events,social commentary,cultural commentary,social commentary,personal development


________________
Target Index: 70
Main String (Query Prompt):
China Needs a Strong Leader Like Xi -- but the Rule of Law Like Singapore
---------------------------------------------------


...

________________
Target Index: 163
Main String (Query Prompt):
The Evolution of the Wedding Invitation from Engraving to Email: Is It a Good Thing?
---------------------------------------------------


,N_5,N_10,N_20,N_40,N_80
Main Key & Tag Position,,,,,
SAE Similarity_1,transformation,transformation,Cultural Commentary,transformation,personal growth
SAE Similarity_2,personal dynamics,guidance,Guidance,personal experience,social justice
SAE Similarity_3,contemporary issues,creativity,Transformation,social commentary,cultural commentary
Activation Similarity_1,evolution,social commentary,Cultural Commentary,social commentary,social commentary
Activation Similarity_2,cultural relevance,transformation,Personal Narratives,personal transformation,personal growth
Activation Similarity_3,societal change,personal narratives,Social Transformation,cultural evolution,cultural evolution
NTK Similarity_1,evolution,evolution,social commentary,social commentary,social commentary
NTK Similarity_2,societal change,social issues,practical advice,personal narratives,personal narratives
NTK Similarity_3,personal experience,exploration,cultural exploration,lifestyle advice,lifestyle advice


________________
Target Index: 189
Main String (Query Prompt):
Debbie Wasserman Schultz Hangs Onto Her Seat In Florida Primary
---------------------------------------------------


,N_5,N_10,N_20,N_40,N_80
Main Key & Tag Position,,,,,
SAE Similarity_1,political maneuvering,politics,politics,politics,politics
SAE Similarity_2,public figures,celebrity,celebrity,celebrity,celebrity
SAE Similarity_3,media discourse,media satire,controversy,social issues,social issues
Activation Similarity_1,celebrity culture,celebrity culture,politics,politics,politics
Activation Similarity_2,personal milestones,current events,celebrity,celebrity,celebrity
Activation Similarity_3,media narratives,personal narratives,social issues,social issues,social issues
NTK Similarity_1,politics,politics,politics,politics,politics
NTK Similarity_2,controversy,celebrity culture,celebrity culture,celebrity culture,social issues
NTK Similarity_3,public figures,social issues,social issues,social issues,celebrity culture


________________


# keep